# Parivar Arth (Family Money Planner) - Exploratory Data Analysis
## Indian CPI Inflation, Category Dynamics, Macro Shocks & RBI Target Band

### Research Objectives:
1. **Trend & Volatility Analysis**: Compare category-specific inflation rates (Food, Housing, Fuel, Transport, Health, Education) against Headline CPI.
2. **RBI Policy Band**: Benchmark historical headline inflation against the Reserve Bank of India's statutory **4% (+/- 2%)** tolerance band (2% to 6%).
3. **Exogenous Transmission**: Analyze lag correlation between international crude oil prices (INR) and domestic transport/fuel inflation.
4. **Structural Shocks**: Examine historical macro shocks: 2020 COVID nationwide lockdown, 2022 Russia-Ukraine global energy shock, and recurring monsoon Kharif rainfall deficits.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Ensure project root is accessible
sys.path.append(os.path.abspath('..'))
from ml.pipeline import build_unified_dataset
from ml.data_sources.mospi_cpi import CATEGORIES

df, meta = build_unified_dataset()
print(f"Loaded {len(df)} monthly observations from {meta['start_date']} to {meta['end_date']}")
print(f"Data Source Status: {'Synthetic Calibrated' if meta['is_synthetic'] else 'Live Official'}")
df.head()

### 1. Headline CPI vs. The RBI 2% - 6% Target Band
Under the flexible inflation targeting framework, the RBI MPC is mandated to keep CPI at 4.0% with a tolerance band between 2.0% and 6.0%.

In [ ]:
fig = go.Figure()

# Upper and Lower RBI Bands
fig.add_trace(go.Scatter(
    x=df['date'], y=[6.0] * len(df),
    mode='lines', line=dict(color='#DC2626', dash='dash'),
    name='RBI Upper Band (6%)'
))
fig.add_trace(go.Scatter(
    x=df['date'], y=[2.0] * len(df),
    mode='lines', line=dict(color='#16A34A', dash='dash'),
    name='RBI Lower Band (2%)'
))
fig.add_trace(go.Scatter(
    x=df['date'], y=[4.0] * len(df),
    mode='lines', line=dict(color='#64748B', dash='dot'),
    name='RBI Target (4%)'
))

# Actual Headline Inflation
fig.add_trace(go.Scatter(
    x=df['date'], y=df['Headline CPI_YoY'],
    mode='lines', line=dict(color='#0F766E', width=2.5),
    name='Headline CPI YoY (%)'
))

fig.update_layout(
    title='Headline CPI Inflation (YoY %) vs. RBI Statutory Band',
    xaxis_title='Year',
    yaxis_title='YoY Inflation Rate (%)',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

### 2. Category-Wise Dispersion: Why a Single Flat Inflation Rate Fails Families
Notice how Education and Healthcare consistently run well above 6%, while Food exhibits extreme periodic spikes.

In [ ]:
fig2 = go.Figure()
palette = ['#0F766E', '#F59E0B', '#2563EB', '#9333EA', '#DC2626', '#059669', '#D97706', '#475569']

for i, cat in enumerate(CATEGORIES):
    col = f"{cat}_YoY"
    if col in df.columns:
        fig2.add_trace(go.Scatter(
            x=df['date'], y=df[col],
            mode='lines',
            name=cat,
            line=dict(color=palette[i % len(palette)], width=1.8)
        ))

fig2.update_layout(
    title='Category-Wise YoY Inflation Dispersion (2015 - 2026)',
    xaxis_title='Year',
    yaxis_title='YoY Inflation Rate (%)',
    template='plotly_white'
)
fig2.show()

### 3. Exogenous Transmission: International Crude in INR vs Domestic Transport
We evaluate how crude oil price shocks in INR transmit to domestic transport inflation with a lag.

In [ ]:
corrs = {}
for lag in range(0, 7):
    shifted_crude = df['crude_inr_pct_change_12m'].shift(lag)
    corrs[f'Lag {lag}M'] = df['Transport and Communication_YoY'].corr(shifted_crude)

lag_df = pd.DataFrame(list(corrs.items()), columns=['Lag Horizon', 'Correlation'])
print('Correlation between Crude Oil (INR) and Domestic Transport Inflation:')
print(lag_df)

px.bar(lag_df, x='Lag Horizon', y='Correlation', title='Transmission Lag: Crude INR % Change to Transport Inflation')

### 4. Summary Findings for Family Budget Planning
1. **Food Volatility**: Food inflation oscillates rapidly depending on Kharif monsoon departures and seasonal perishables, necessitating a flexible grocery buffer.
2. **Sticky Social Costs**: Healthcare (~6.5-7.5%) and Education (~7-8%) have virtually zero downward elasticity, meaning family sinking funds must compound at higher rates.
3. **Energy Shocks**: Spikes in international crude pass through to transport within 1-2 months, immediately eroding disposable surpluses for commuting households.